# Open Road Risk GIS export for QGIS

Open Road Risk is an open-data road safety pipeline that estimates exposure-adjusted collision risk across the GB road network. This notebook demonstrates the QGIS-ready GeoPackage export: a spatial layer with one row per OS Open Roads link, combining road geometry, estimated traffic exposure, observed injury-collision summaries, and modelled risk ranking fields.

The public Kaggle Dataset is https://www.kaggle.com/datasets/thomassimm/open-road-risk-gb-link-risk-exposure-gis/.

The main project site is https://openroadrisk.org/ and the source repository is https://github.com/ThomasHSimm/open-road-risk.

This Kaggle notebook uses an already-built GeoPackage attached as a Kaggle Dataset. The canonical export builder remains `src/road_risk/outputs/gis_link_export.py` in the repository. This notebook does not rebuild the export, rerun modelling, rerun STATS19 processing, rerun snapping/joining, rebuild traffic/AADT, or rescore risk.

Kaggle renders the plots when the notebook is run; outputs are cleared in the repository copy to keep the notebook lightweight.

This is a screening and research dataset, not causal proof or a road safety engineering audit.

## How to cite

If you use this dataset, project outputs, or related Open Road Risk materials, please cite:

Simm, T. H. (2026). *Open Road Risk: an open-data pipeline for exposure-adjusted collision risk across the Great Britain road network*. Zenodo. https://doi.org/10.5281/zenodo.20451731


## Licence and attribution

This GeoPackage is a derived export from multiple open-data sources and Open Road Risk processing/model outputs. It should not be treated as having one simple standalone licence that overrides upstream terms. Users are responsible for preserving applicable attribution, copyright, database-right, and licence requirements for the source data used in any onward use or redistribution.

Suggested attribution text:

- Contains OS data © Crown copyright and database right.
- Contains public sector information licensed under the Open Government Licence v3.0.
- Contains OpenStreetMap-derived information where OSM-derived features are used; OpenStreetMap data is available under the Open Database Licence.
- Open Road Risk processing, modelling code, and export logic: Thomas H. Simm / Open Road Risk.


## Setup paths

Attach the public Kaggle Dataset containing `open-road-risk-gb-link-risk-exposure.gpkg`. The lookup below supports the usual Kaggle mount path, the `/datasets/{owner}/{slug}` path, and a recursive fallback below `/kaggle/input`.


In [ ]:
from pathlib import Path

GPKG_NAME = "open-road-risk-gb-link-risk-exposure.gpkg"
LAYER = "gb_link_risk_exposure"

candidate_roots = [
    Path("/kaggle/input/open-road-risk-gb-link-risk-exposure-gis"),
    Path("/kaggle/input/datasets/thomassimm/open-road-risk-gb-link-risk-exposure-gis"),
    Path("/kaggle/input"),
]

matches = []
for root in candidate_roots:
    if root.exists():
        direct = root / GPKG_NAME
        if direct.exists():
            matches.append(direct)
        matches.extend(sorted(root.rglob(GPKG_NAME)))

seen = set()
matches = [path for path in matches if not (path in seen or seen.add(path))]
if not matches:
    raise FileNotFoundError(f"Could not find {GPKG_NAME} below /kaggle/input")

GPKG = matches[0]
PACKAGE_DIR = Path("/kaggle/working/open-road-risk-gis-export")
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

print("GeoPackage:", GPKG)
print("Working package directory:", PACKAGE_DIR)


## Dependency check

Kaggle images usually include the data stack, but geospatial libraries vary by image version. This cell installs only missing packages.

In [ ]:
import importlib.util
import subprocess
import sys

required = ["geopandas", "matplotlib", "pandas", "pyogrio", "shapely"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

print("Missing dependencies installed:" if missing else "Dependencies available", missing)

## Fast GeoPackage metadata check

GeoPackage is a spatial database format. A single `.gpkg` can contain one or more layers, each with geometry, typed attributes, CRS metadata, and spatial indexing.

This notebook first reads metadata rather than loading 3.94M geometries into memory. `pyogrio.read_info()` is a fast way to inspect the layer name, CRS, fields, geometry type, bounds, and feature count.

The row count checks full-network coverage, the bounds confirm the GB extent, and EPSG:4326 means the layer is straightforward to use in web/GIS workflows.

In [ ]:
import pandas as pd
import pyogrio

info = pyogrio.read_info(GPKG, layer=LAYER)

metadata_summary = pd.DataFrame(
    [
        ("Layer", info["layer_name"]),
        ("Row count", f"{info['features']:,}"),
        ("CRS", info["crs"]),
        ("Geometry type", info["geometry_type"]),
        ("Bounds", info["total_bounds"]),
        ("File size GB", round(GPKG.stat().st_size / 1024**3, 3)),
    ],
    columns=["check", "value"],
)
metadata_summary

## Attribute completeness and country coverage

This reads selected non-geometry columns only. Non-null counts show which risk/exposure fields are complete. Country counts provide a quick England/Wales/Scotland coverage check through the `deprivation_country` assignment.

In [ ]:
key_fields = [
    "risk_percentile",
    "predicted_xgb",
    "predicted_eb",
    "estimated_aadt",
    "exposure_vehicle_km_year",
    "crude_rate_per_million_vkm",
    "deprivation_country",
]

available_fields = set(info["fields"])
read_fields = [field for field in key_fields if field in available_fields]

attrs = pyogrio.read_dataframe(
    GPKG,
    layer=LAYER,
    columns=read_fields,
    read_geometry=False,
)

non_null = attrs[read_fields].notna().sum().rename("non_null_count").to_frame()
non_null["row_count"] = len(attrs)
non_null["non_null_pct"] = (non_null["non_null_count"] / len(attrs) * 100).round(2)
non_null

In [ ]:
if "deprivation_country" in attrs.columns:
    country_counts = (
        attrs["deprivation_country"]
        .fillna("missing")
        .value_counts()
        .rename_axis("deprivation_country")
        .reset_index(name="link_count")
    )
    country_counts["share_pct"] = (country_counts["link_count"] / len(attrs) * 100).round(2)
    display(country_counts)
else:
    print("deprivation_country is not present in this GeoPackage.")

## What is inside the layer?

The preview below reads ten rows with geometry. This is enough to show the layer behaves like a normal GIS dataset without loading the full network into memory.

In [ ]:
preview_columns = [
    "link_id",
    "risk_percentile",
    "risk_decile",
    "is_top_1pct",
    "estimated_aadt",
    "link_length_km",
    "exposure_vehicle_km_year",
    "crude_rate_per_million_vkm",
    "collision_count",
    "predicted_xgb",
    "road_classification",
    "deprivation_country",
]

preview = pyogrio.read_dataframe(
    GPKG,
    layer=LAYER,
    columns=preview_columns,
    max_features=10,
)
preview

## Column inventory

The inventory combines GeoPackage field metadata, non-null counts, and short meanings for the main analytical fields. The counts use SQLite aggregate queries against the GeoPackage, so the notebook does not need to load the full geometry layer.

In [ ]:
import sqlite3

field_meanings = {
    "risk_percentile": "Relative modelled risk ranking across scored GB links; higher means higher ranked risk.",
    "risk_decile": "risk_percentile grouped into deciles; 10 is highest.",
    "estimated_aadt": "Estimated annual average daily traffic.",
    "exposure_vehicle_km_year": "Estimated annual vehicle-km exposure: estimated_aadt x link_length_km x 365.",
    "crude_rate_per_million_vkm": "Observed collisions per million vehicle-km; not the modelled risk score.",
    "predicted_xgb": "Modelled expected collision score/count from XGBoost.",
    "predicted_eb": "Empirical Bayes adjusted score where available.",
    "collision_count": "Observed retained injury collisions on the link.",
    "is_top_1pct": "True for links with risk_percentile >= 99.",
    "is_top_5pct": "True for links with risk_percentile >= 95.",
    "is_top_decile": "True for links with risk_percentile >= 90.",
    "geometry": "OS Open Roads link geometry in EPSG:4326.",
}

def quote_identifier(name: str) -> str:
    return '"' + name.replace('"', '""') + '"'

geometry_column = info.get("geometry_name", "geom")
count_columns = list(info["fields"]) + [geometry_column]
non_null_by_column = {}

with sqlite3.connect(GPKG) as conn:
    layer_sql = quote_identifier(LAYER)
    for column in count_columns:
        column_sql = quote_identifier(str(column))
        non_null_by_column[str(column)] = conn.execute(
            f"SELECT COUNT({column_sql}) FROM {layer_sql}"
        ).fetchone()[0]

inventory = pd.DataFrame({"column": list(info["fields"]), "dtype": list(info["dtypes"])})
inventory.loc[len(inventory)] = ["geometry", info["geometry_type"]]
non_null_by_column["geometry"] = non_null_by_column.get(str(geometry_column))
inventory["non_null_count"] = inventory["column"].map(non_null_by_column).astype("Int64")
inventory["meaning"] = inventory["column"].map(field_meanings).fillna("")
inventory

## National high-risk pattern

This plot reads only the top 1% and top 0.1% risk-ranked links. It is a screening/ranking view of the national pattern, not a proof that any particular link has a dangerous design.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

plot_cols = [
    "link_id",
    "risk_percentile",
    "risk_decile",
    "deprivation_country",
]

top_1 = pyogrio.read_dataframe(
    GPKG,
    layer=LAYER,
    columns=plot_cols,
    where="risk_percentile >= 99.0",
)

top_01 = pyogrio.read_dataframe(
    GPKG,
    layer=LAYER,
    columns=plot_cols,
    where="risk_percentile >= 99.9",
)

assert isinstance(top_1, gpd.GeoDataFrame)
assert isinstance(top_01, gpd.GeoDataFrame)

fig, ax = plt.subplots(figsize=(8, 10))
top_1.plot(ax=ax, color="#d9d9d9", linewidth=0.25, alpha=0.35)
top_01.plot(
    ax=ax,
    column="risk_percentile",
    cmap="magma_r",
    linewidth=0.8,
    legend=True,
)
ax.set_title("Open Road Risk: highest-ranked GB road links", fontsize=14)
ax.set_axis_off()
plt.show()

## Leeds local GIS example

This close-up shows how the GeoPackage can support local inspection. It reads a smaller central Leeds bounding box and plots all links in that window, coloured by modelled risk percentile.


In [ ]:
leeds_bbox = (-1.62, 53.75, -1.48, 53.84)

context_cols = [
    "link_id",
    "risk_percentile",
    "risk_decile",
    "road_classification",
]

leeds_all = pyogrio.read_dataframe(
    GPKG,
    layer=LAYER,
    columns=context_cols,
    bbox=leeds_bbox,
)

fig, ax = plt.subplots(figsize=(8, 8))
leeds_all.plot(
    ax=ax,
    column="risk_percentile",
    cmap="magma_r",
    linewidth=0.55,
    alpha=0.95,
    legend=True,
)
ax.set_title("Leeds example: all local links by modelled risk percentile", fontsize=14)
ax.set_xlim(leeds_bbox[0], leeds_bbox[2])
ax.set_ylim(leeds_bbox[1], leeds_bbox[3])
ax.set_axis_off()
plt.show()

## How to read these views

The national plot is a screening and prioritisation view. The Leeds plot shows that the same GeoPackage can support local GIS inspection without loading the full network.

High-risk links are candidates for review, not proof of dangerous design. Modelled risk and crude collision rate are different quantities: modelled risk combines exposure, road context, and learned patterns, while the crude rate is a simple observed-count ratio that can be unstable on short links or sparse collision histories.

For full-network interactive exploration, QGIS is the better tool: load the GeoPackage layer, filter by `risk_decile` or `is_top_1pct`, and apply the included QGIS styles if they are present in the dataset.

## Checksums and optional package manifest

This writes `checksums.txt` in Kaggle working storage for the attached GeoPackage and any optional README/QML files found beside it. The GeoPackage is not copied by default, avoiding a second large file in `/kaggle/working`.

In [ ]:
import hashlib
import shutil

optional_names = ["README.md", "risk_decile.qml", "crude_collision_rate.qml"]
checksum_files = [GPKG]

for name in optional_names:
    candidates = [GPKG.parent / name, INPUT_ROOT / name]
    source = next((path for path in candidates if path.exists()), None)
    if source is None:
        continue
    target = PACKAGE_DIR / name
    if source.resolve() != target.resolve():
        shutil.copy2(source, target)
    checksum_files.append(target)

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

with (PACKAGE_DIR / "checksums.txt").open("w", encoding="utf-8") as handle:
    for path in checksum_files:
        handle.write(f"{sha256(path)}  {path.name}\n")

print("Checksum manifest:", PACKAGE_DIR / "checksums.txt")
print((PACKAGE_DIR / "checksums.txt").read_text(encoding="utf-8"))